In [14]:
#setup
!pip install -q google-genai pypdf

from google import genai
from google.genai import types
import numpy as np
from google.colab import userdata

client = genai.Client(api_key=userdata.get('Ragproject'))

In [15]:
#uploading a file
from google.colab import files
upload = files.upload()

pdf_name=list(upload.keys())[0]
print("Uploaded pdf file: ",pdf_name)

Saving NETMate_Research_Aptitude_Notes.pdf to NETMate_Research_Aptitude_Notes (1).pdf
Uploaded pdf file:  NETMate_Research_Aptitude_Notes (1).pdf


In [16]:
#extracting text
from pypdf import PdfReader
reader=PdfReader(pdf_name)
print("Total number of pages in the pdf: ",len(reader.pages))

text=''
for pg in reader.pages:
  text+=pg.extract_text() + "\n"

print("Total umber of characters in the pdf: ",len(text))

Total number of pages in the pdf:  3
Total umber of characters in the pdf:  6584


In [17]:
#chunking

def chunking(text,chunk_size=1000,overlap=150):
  chunk=[]
  start=0
  while(start<len(text)):
    end=start+chunk_size
    chunk.append(text[start:end])
    start=end-overlap
  return chunk

chunks=chunking(text)
print("Total number of chunks: ",len(chunks))


Total number of chunks:  8


In [18]:
print(chunks[0][-150:])
print()
print(chunks[1][:150])

tal (basic) research: It aims to expand knowledge and build theory. It has no immediate
practical use. Example: studying how memory works in the human

tal (basic) research: It aims to expand knowledge and build theory. It has no immediate
practical use. Example: studying how memory works in the human


In [26]:
#embedding

EMBEDDING_MODEL='gemini-embedding-001'
EMBEDDING_DIMENSION=768

def embedding(t):
  response=client.models.embed_content(
      model=EMBEDDING_MODEL,
      contents=t,
      config=types.EmbedContentConfig(
          output_dimensionality=EMBEDDING_DIMENSION
      )
  )
  return(response.embeddings[0].values)


embeded_chunks=[]
for i, ch in enumerate(chunks):
  embeded_chunks.append(np.array(embedding(ch)))
  print(f"Embeded chunks {i+1}/{len(chunks)}")

embeded_chunks=np.array(embeded_chunks)
print("Embeded chunks shape: ",embeded_chunks.shape)

#Normalization

embeded_chunks=embeded_chunks/np.linalg.norm(embeded_chunks, axis=1, keepdims=True)
print("Normalized Embeded chunks shape: ",embeded_chunks.shape)


Embeded chunks 1/8
Embeded chunks 2/8
Embeded chunks 3/8
Embeded chunks 4/8
Embeded chunks 5/8
Embeded chunks 6/8
Embeded chunks 7/8
Embeded chunks 8/8
Embeded chunks shape:  (8, 768)
Normalized Embeded chunks shape:  (8, 768)


In [32]:
#retrieve function, QUESTION EMBEDDING

def retrieve(question,top_k=2):
  q_vector=np.array(embedding(question))
  q_vector=q_vector/np.linalg.norm(q_vector)

  score = embeded_chunks @ q_vector

  top_result = np.argsort(score)[::-1][:top_k]

  return((chunks[i], score[i]) for i in top_result)



#testing retrieve

question = retrieve("What is stratified sampling?")

for chunk, score in question:
  print(f"score: {score:.4f}")
  print(chunk)
  print()


score: 0.6975
e study. Sampling is done to save time and cost.
Probability sampling gives every member of the population a known chance of being selected. Its
types are: simple random sampling (every member has an equal chance, like a lottery), systematic
sampling (selecting every kth member from a list), stratified sampling (the population is divided into
homogeneous groups called strata, and a random sample is taken from each stratum), and cluster
sampling (the population is divided into groups called clusters, and whole clusters are randomly
selected).
Non-probability sampling does not give every member a known chance. Its types are: convenience
sampling (choosing whoever is easily available), purposive or judgemental sampling (the researcher
picks members deliberately for a purpose), quota sampling (fixed numbers are taken from each
category), and snowball sampling (existing participants recruit new participants, useful for hard-to-find
groups).
Probability sampling is preferred wh

In [36]:
#generating answer

MODEL='gemini-3.1-flash-lite'



configuration = types.GenerateContentConfig(
    system_instruction=(
        "You are NETMate, a study assistant for UGC NET Paper 1 that answers only from the study notes provided in the context. "
        "Explain in simple English suitable for an exam aspirant. "
        "If the answer is not in the context, reply exactly: "
        "I could not find this in the notes. "
        "Never guess or use outside knowledge."
    ),
    temperature=0.3
)



def ans(question, top_k=2):
  top=retrieve(question,top_k)

  content="\n\n_______\n\n".join(chunk for chunk, score in top)

  prompt = f"""Context from the study notes:
      {content}
      QUESTION: {question}
  Answer using only the context above."""

  response=client.models.generate_content(
      model=MODEL,
      contents=prompt,
      config=configuration
  )
  return response.text


print(ans('What is the difference between fundamental and applied research?'))

The difference between fundamental and applied research is as follows:

*   **Fundamental (basic) research:** Its goal is to expand knowledge and build theory. It does not have any immediate practical use. (Example: studying how memory works in the human brain).
*   **Applied research:** It uses existing knowledge to solve a specific practical problem. (Example: finding the best teaching method to improve reading skills in Class 5).


In [37]:
print(ans('What is stratified sampling?'))

Stratified sampling is a type of probability sampling where the population is divided into homogeneous groups called strata, and a random sample is then taken from each of these strata.


In [38]:
print(ans('What is a Type I error?'))

A Type I error is rejecting a null hypothesis that is actually true (a false positive). The probability of this error is called the level of significance, written as alpha, and is commonly set at 0.05.


In [40]:
print(ans('Who is the PM of India?'))

I could not find this in the notes.
